# 8.3 File Handling — JSON

**Prerequisites:** 8.1 File Handling — Text, 2.5 Dictionary  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What JSON is, and the Python ↔ JSON type mapping
- `dump` / `dumps` / `load` / `loads` — and the `s` mnemonic
- Formatting: `indent`, `sort_keys`, `ensure_ascii`
- 🔴 **What JSON cannot represent** — `datetime`, `Decimal`, `set`, tuples
- Custom encoders (`default=`) and decoders (`object_hook=`)
- Handling `JSONDecodeError`
- **JSON Lines** for streaming and appending
- Why JSON is safe to parse and `pickle` is not

---

## JSON files

**JavaScript Object Notation** is a lightweight, text-based data-interchange format. It
encodes Python objects as JSON strings, and decodes JSON strings back into Python objects.

It is the default format for web APIs, configuration files, and any time two programs that
may be written in different languages need to exchange structured data.

### Why JSON won

| | JSON | `pickle` | CSV |
|---|---|---|---|
| Human-readable | ✅ | ❌ | ✅ |
| Cross-language | ✅ | ❌ Python only | ✅ |
| Nested structures | ✅ | ✅ | ❌ flat only |
| Safe to parse untrusted input | ✅ | 🔴 **No — executes code** | ✅ |
| Preserves Python types exactly | ❌ | ✅ | ❌ |

That fourth row is the important one. `pickle.load()` on a file you did not create is
equivalent to running an untrusted script (**7.1**). `json.loads()` on hostile input can
produce bad *data*, but it cannot execute anything.

### The four functions

| Function | Direction | Target |
|---|---|---|
| `json.dump(obj, file)` | Python → JSON | a **file** |
| `json.dumps(obj)` | Python → JSON | a **string** |
| `json.load(file)` | JSON → Python | from a **file** |
| `json.loads(s)` | JSON → Python | from a **string** |

> **The mnemonic:** the **`s`** stands for **string**. `dumps` dumps *to a string*; `loads`
> loads *from a string*. Without the `s`, it is a file.

In [ ]:
import json

data = {
    "name": "Aditya",
    "age": 30,
    "languages": ["Python", "SQL"],
    "active": True,
    "manager": None,
    "score": 91.5,
}

# ---- dumps: Python -> JSON string ----
compact = json.dumps(data)
print("dumps (compact):")
print(" ", compact)
print("  type:", type(compact).__name__)

# ---- loads: JSON string -> Python ----
restored = json.loads(compact)
print("\nloads:")
print(" ", restored)
print("  round trip identical:", restored == data)

# ---- Formatting for humans ----
print("\ndumps(indent=2, sort_keys=True):")
print(json.dumps(data, indent=2, sort_keys=True))

# ---- Separators: control the compactness precisely ----
print("\nsmallest possible (no spaces):")
print(" ", json.dumps(data, separators=(",", ":")))

# ---- ensure_ascii: keep non-ASCII readable ----
multilingual = {"greeting": "नमस्कार", "city": "München"}
print("\ndefault (ensure_ascii=True):")
print(" ", json.dumps(multilingual))
print("ensure_ascii=False:")
print(" ", json.dumps(multilingual, ensure_ascii=False))
print("  ^ both are valid JSON; the second is far easier to read")

### The type mapping

`json` translates between Python types and JSON types. The translation is **lossy in one
direction** — several distinct Python types collapse onto the same JSON type.

| Python | → JSON | JSON | → Python |
|---|---|---|---|
| `dict` | `object` | `object` | `dict` |
| `list`, **`tuple`** | `array` | `array` | **`list`** |
| `str` | `string` | `string` | `str` |
| `int` | `number` | `number` (int) | `int` |
| `float` | `number` | `number` (real) | `float` |
| `True` / `False` | `true` / `false` | `true` / `false` | `True` / `False` |
| `None` | `null` | `null` | `None` |

### 🔴 The two lossy conversions

1. **A `tuple` becomes a `list`.** Serialise `(1, 2)` and you get `[1, 2]` back — a *list*.
   Round-tripping does not preserve the type.
2. **Dict keys are always strings.** `{1: "a"}` becomes `{"1": "a"}`. The integer key comes
   back as the string `"1"`.

Both are silent. Neither raises.

In [ ]:
import json

# ---- 🔴 tuple -> list, silently ----
original = {"point": (3, 4), "tags": ("a", "b")}
round_tripped = json.loads(json.dumps(original))

print("before:", original)
print("after :", round_tripped)
print("equal :", original == round_tripped, " <- tuples became lists")
print("type  :", type(round_tripped["point"]).__name__)


# ---- 🔴 non-string keys -> string keys, silently ----
numbered = {2: "int key", 3.5: "float key", True: "bool key"}
back = json.loads(json.dumps(numbered))
print("\nbefore:", numbered)
print("after :", back)
print("  ^ every key is now a string: 2 -> '2', 3.5 -> '3.5', True -> 'true'")


# ---- ⚠️ True == 1, so they collide as dict keys BEFORE json is involved ----
collided = {1: "first", True: "bool key"}
print("\n{1: 'first', True: 'bool key'} ->", collided)
print("  ^ one entry, not two: True == 1, so the second assignment kept the")
print("    original key 1 and overwrote its VALUE - pure dict behaviour (2.5)")


# ---- 🔴 Types JSON cannot represent at all ----
from datetime import datetime, date, timezone
from decimal import Decimal

unsupported = [
    ("set", {1, 2, 3}),
    ("datetime", datetime(2024, 3, 15, tzinfo=timezone.utc)),
    ("date", date(2024, 3, 15)),
    ("Decimal", Decimal("19.99")),
    ("bytes", b"raw"),
    ("complex", 3 + 4j),
]

print()
for label, value in unsupported:
    try:
        json.dumps(value)
        print(f"  {label:<10} serialised")
    except TypeError as exc:
        print(f"  {label:<10} TypeError: {exc}")

print("""
These at least FAIL LOUDLY, which is better than the silent tuple/key
conversions above. The next section shows how to handle them.
""")

### Custom encoding: `default=` and `object_hook=`

For the types JSON cannot represent, you supply the translation.

**Writing** — pass `default=`, a function called for any object `json` cannot handle:

```python
json.dumps(obj, default=my_encoder)
```

**Reading** — pass `object_hook=`, a function called on every decoded JSON object, giving
you a chance to reconstruct richer types:

```python
json.loads(text, object_hook=my_decoder)
```

**Real-world use case:** almost every API payload contains timestamps. Getting `datetime`
in and out of JSON cleanly is the single most common reason to reach for these.

In [ ]:
import json
from datetime import datetime, date, timezone
from decimal import Decimal


# ---- Writing: a `default` function for the awkward types ----
def encode_extras(obj):
    """Called by json for anything it cannot serialise itself."""
    if isinstance(obj, datetime):
        return {"__type__": "datetime", "value": obj.isoformat()}
    if isinstance(obj, date):
        return {"__type__": "date", "value": obj.isoformat()}
    if isinstance(obj, Decimal):
        return {"__type__": "decimal", "value": str(obj)}
    if isinstance(obj, set):
        return {"__type__": "set", "value": sorted(obj)}
    raise TypeError(f"cannot serialise {type(obj).__name__}")


order = {
    "id": "A-1001",
    "placed_at": datetime(2024, 3, 15, 14, 30, tzinfo=timezone.utc),
    "delivery": date(2024, 3, 18),
    "total": Decimal("19.99"),
    "tags": {"priority", "gift"},
}

encoded = json.dumps(order, default=encode_extras, indent=2)
print(encoded)


# ---- Reading: an `object_hook` to reverse it ----
def decode_extras(mapping):
    """Called on every JSON object; rebuild the richer type if we tagged it."""
    kind = mapping.get("__type__")
    if kind == "datetime":
        return datetime.fromisoformat(mapping["value"])
    if kind == "date":
        return date.fromisoformat(mapping["value"])
    if kind == "decimal":
        return Decimal(mapping["value"])
    if kind == "set":
        return set(mapping["value"])
    return mapping


decoded = json.loads(encoded, object_hook=decode_extras)

print("\nafter round trip:")
for key, value in decoded.items():
    print(f"  {key:<11} {value!r:<45} {type(value).__name__}")

print("\nfully restored:", decoded == order)


# ---- The simpler option when you only need ONE direction ----
simple = json.dumps(
    {"placed_at": datetime(2024, 3, 15, tzinfo=timezone.utc)},
    default=str,                       # str() anything unknown
)
print("\ndefault=str:", simple)
print("  ^ fine for logs and display; not round-trippable")

### Reading and writing files

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path

# Scratch directory - this notebook never writes into the repository (8.1, 8.5)
work_dir = Path(tempfile.mkdtemp(prefix="py83_json_"))
config_path = work_dir / "config.json"

settings = {
    "service": "payments",
    "timeout": 30,
    "retries": 3,
    "endpoints": {
        "base": "https://api.example.com",
        "health": "/healthz",
    },
    "features": ["retry", "circuit-breaker"],
}

# ---- dump: write to a FILE ----
# Note encoding="utf-8" - same rule as 8.1
with open(config_path, "w", encoding="utf-8") as handle:
    json.dump(settings, handle, indent=2)

print("written:", config_path.stat().st_size, "bytes\n")
print(config_path.read_text(encoding="utf-8"))

# ---- load: read from a FILE ----
with open(config_path, encoding="utf-8") as handle:
    loaded = json.load(handle)

print("timeout   :", loaded["timeout"])
print("base url  :", loaded["endpoints"]["base"])
print("round trip:", loaded == settings)

# ---- pathlib one-liners, for whole small files ----
config_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
again = json.loads(config_path.read_text(encoding="utf-8"))
print("\nvia pathlib:", again == settings)

shutil.rmtree(work_dir, ignore_errors=True)   # tidy up

### 🔴 Handling malformed JSON

Anything you did not write yourself can be malformed — a truncated download, a hand-edited
config, an API returning an HTML error page instead of JSON.

`json.JSONDecodeError` is a subclass of `ValueError`, and it carries **`msg`, `pos`, `lineno`
and `colno`** — enough to point at exactly where the parse failed.

In [ ]:
import json

broken_inputs = {
    "trailing comma":  '{"a": 1, "b": 2,}',
    "single quotes":   "{'a': 1}",
    "unquoted key":    '{a: 1}',
    "truncated":       '{"a": 1, "b":',
    "python literals": '{"a": True, "b": None}',
    "an HTML page":    '<!DOCTYPE html><html>500 Server Error</html>',
    "empty":           '',
}

for label, text in broken_inputs.items():
    try:
        json.loads(text)
        print(f"  {label:<16} parsed OK")
    except json.JSONDecodeError as exc:
        print(f"  {label:<16} line {exc.lineno} col {exc.colno}: {exc.msg}")

print("""
Note 'python literals': JSON uses true/false/null, not True/False/None.
This bites when someone builds JSON with str(dict) instead of json.dumps().
""")


# ---- The defensive pattern ----
def load_config(text: str, default: dict | None = None) -> dict:
    """Parse JSON, returning `default` (not raising) on malformed input."""
    try:
        result = json.loads(text)
    except json.JSONDecodeError as exc:
        print(f"  invalid JSON at line {exc.lineno}, col {exc.colno}: {exc.msg}")
        return default or {}

    if not isinstance(result, dict):
        print(f"  expected an object, got {type(result).__name__}")
        return default or {}
    return result


print("valid  :", load_config('{"timeout": 30}'))
print("broken :", load_config('{"timeout": }', default={"timeout": 10}))
print("wrong shape:", load_config('[1, 2, 3]'))

# ⚠️ JSONDecodeError IS a ValueError, so `except ValueError` catches it too
print("\nis a ValueError:", issubclass(json.JSONDecodeError, ValueError))

### JSON Lines (`.jsonl`)

A single JSON document has a problem for logs and data feeds: you cannot **append** to it
without rewriting the whole file, and you cannot read it without parsing all of it.

**JSON Lines** solves this — *one complete JSON object per line*:

```
{"ts": "2024-03-15T09:00:00Z", "level": "INFO",  "msg": "started"}
{"ts": "2024-03-15T09:01:00Z", "level": "ERROR", "msg": "db timeout"}
```

| | JSON array | JSON Lines |
|---|---|---|
| Append a record | Rewrite the whole file | `handle.write(line)` |
| Read one record | Parse everything | Parse one line |
| Streams | ❌ | ✅ |
| A corrupt record | Breaks the whole file | Breaks one line |

**Real-world use case:** application logs, ML training data, database exports, and the
response format of many streaming APIs.

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path

# Scratch directory again - nothing is written into the repository
work_dir = Path(tempfile.mkdtemp(prefix="py83_jsonl_"))
log_path = work_dir / "events.jsonl"

events = [
    {"ts": "2024-03-15T09:00:00Z", "level": "INFO", "msg": "service started"},
    {"ts": "2024-03-15T09:01:00Z", "level": "ERROR", "msg": "db timeout"},
    {"ts": "2024-03-15T09:02:00Z", "level": "INFO", "msg": "retry scheduled"},
]

# ---- Writing: one object per line, appendable ----
with open(log_path, "w", encoding="utf-8") as handle:
    for event in events:
        handle.write(json.dumps(event) + "\n")

# Appending later costs nothing
with open(log_path, "a", encoding="utf-8") as handle:
    handle.write(json.dumps({"ts": "2024-03-15T09:03:00Z",
                             "level": "ERROR", "msg": "disk full"}) + "\n")

print(log_path.read_text(encoding="utf-8"))

# ---- Reading: stream, one record at a time ----
errors = []
with open(log_path, encoding="utf-8") as handle:
    for line_no, line in enumerate(handle, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            print(f"  line {line_no} is corrupt, skipping: {exc.msg}")
            continue
        if record["level"] == "ERROR":
            errors.append(record["msg"])

print("errors found:", errors)

# ---- One bad line does not destroy the file ----
with open(log_path, "a", encoding="utf-8") as handle:
    handle.write("{this is not valid json}\n")
    handle.write(json.dumps({"ts": "09:04", "level": "INFO", "msg": "recovered"}) + "\n")

good = 0
with open(log_path, encoding="utf-8") as handle:
    for line in handle:
        try:
            json.loads(line)
            good += 1
        except json.JSONDecodeError:
            pass

print(f"\n{good} of 6 lines still parse - the corruption is contained")
shutil.rmtree(work_dir, ignore_errors=True)   # tidy up

---

## Common Mistakes & Pitfalls

1. 🔴 **Assuming a tuple round-trips.** `(1, 2)` becomes `[1, 2]` — a *list* — silently.
2. 🔴 **Assuming dict keys keep their type.** `{1: 'a'}` becomes `{'1': 'a'}`.
3. **Trying to serialise a `datetime`, `Decimal`, `set` or `bytes`** without a `default=` function — `TypeError`.
4. **Building JSON with `str(my_dict)`.** That produces Python syntax (`True`, `None`, single quotes), which is **not** valid JSON. Use `json.dumps()`.
5. **Not catching `JSONDecodeError`** on input you did not create. An API returning an HTML error page is a very common cause.
6. **Omitting `encoding="utf-8"`** when opening the file (**8.1**).
7. **Using `json.load()` on a huge file** when JSON Lines would let you stream.
8. **Reaching for `pickle` because JSON 'cannot handle' your object.** Write a `default=` function instead — `pickle` on untrusted input executes code.
9. **Assuming JSON preserves key order.** On the Python side it does — dict insertion order is a *language guarantee* since 3.7, not a CPython implementation detail — but the JSON *spec* does not guarantee it, so other tools and languages may reorder keys.

## Best Practices

- Use `json.dumps(obj, indent=2)` for anything a human will read; compact for the wire.
- Pass `ensure_ascii=False` when the data contains non-ASCII text you want readable.
- Always `encoding="utf-8"` on the file handle.
- Wrap `json.loads()` in `try/except json.JSONDecodeError` for external input.
- **Validate the shape** after parsing — valid JSON is not the same as valid data.
- Serialise `datetime` as **ISO 8601** (`.isoformat()`), which is what every other language expects.
- Use **JSON Lines** for logs, exports and anything appended to over time.
- Prefer JSON over `pickle` for anything crossing a trust or language boundary.
- For strict schema validation, look at `pydantic` (**18**).

## Practice Exercises

Try these before moving on.

1. Round-trip a dict containing a tuple and an integer key. Explain both changes.
2. Write a `default=` function that serialises `datetime`, `Decimal` and `set`, and an `object_hook` that restores them.
3. Parse five malformed JSON strings and report the line and column of each failure.
4. Write a config loader that returns defaults on malformed JSON and validates that the result is an object with the expected keys.
5. Convert a CSV from **8.2** into JSON Lines, then read it back and aggregate a column.
6. Compare the byte size of `indent=2` vs `separators=(',',':')` for a 100-record list.
7. Given a nested API response, write `get_nested()` (**2.7**) to pull a deep value safely.
8. Explain to someone why `json.loads()` is safe on untrusted input but `pickle.load()` is not.